In [14]:
"""
GPU DM-Soliton Simulator | v4.3-Referee-Proof-Panel (JCP Submission Ready)
================================================================================
PAPER TITLE: Eliminating Spurious First-Order Convergence in Split-Step Fourier
             Simulations of Dispersion-Managed Solitons

ALL FIGURES MERGED INTO PANELS -- No empty subplots, 1x3 / 2x3 layouts.
"""

import os
import time
import numpy as np
import cupy as cp
from tqdm import tqdm
from scipy.stats import linregress
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.font_manager as fm
import warnings

# ==============================================================================
# 0. OUTPUT DIRECTORY
# ==============================================================================
FIG_DIR = "./figures_jcp"
os.makedirs(FIG_DIR, exist_ok=True)

def figpath(fname):
    base = os.path.splitext(fname)[0]
    return os.path.join(FIG_DIR, base + ".pdf")

# ==============================================================================
# 1. JOURNAL-QUALITY STYLE (JCP / AIP standard)
# ==============================================================================
def _configure_fonts():
    available = {f.name for f in fm.fontManager.ttflist}
    preferred = ["Times New Roman", "DejaVu Serif", "Liberation Serif",
                 "Nimbus Roman", "Computer Modern Roman", "Noto Serif"]
    found = [f for f in preferred if f in available]
    serif_fallback = found if found else ["serif"]
    plt.rcParams.update({
        "font.family":        serif_fallback,
        "font.size":          9,
        "axes.labelsize":     9,
        "xtick.labelsize":    8,
        "ytick.labelsize":    8,
        "legend.fontsize":    7.5,
        "axes.titlesize":     9,
        "lines.linewidth":    1.2,
        "lines.markersize":   5,
        "patch.linewidth":    0.8,
        "axes.linewidth":     0.8,
        "axes.spines.top":    True,
        "axes.spines.right":  True,
        "xtick.direction":    "in",
        "ytick.direction":    "in",
        "xtick.major.size":   4.0,  "xtick.minor.size":  2.0,
        "ytick.major.size":   4.0,  "ytick.minor.size":  2.0,
        "xtick.major.width":  0.8,  "xtick.minor.width": 0.6,
        "ytick.major.width":  0.8,  "ytick.minor.width": 0.6,
        "xtick.top":          True,
        "ytick.right":        True,
        "figure.dpi":         150,
        "savefig.dpi":        600,
        "savefig.bbox":       "tight",
        "savefig.pad_inches": 0.02,
        "pdf.fonttype":       42,
        "ps.fonttype":        42,
    })
    print(f"[Font] Serif family: {serif_fallback[0]}")
    if "Times New Roman" not in available:
        print("[Font] 'Times New Roman' not found; graceful fallback applied.")

_configure_fonts()
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

# Figure dimensions (JCP single/double column)
W1 = 3.375          # single column, inches
W2 = 6.75           # double column
H1 = W1 / 1.618     # golden ratio ~2.08 in
H2 = W2 / 1.618     # ~4.17 in

# Colorblind-safe palette
C_TRAD  = "#C0392B"
C_MID   = "#2166AC"
C_SS4   = "#1A9641"
C_REF1  = "#888888"
C_REF2  = "#333333"
C_REF4  = "#762A83"

def _log_minor_ticks(ax):
    for axis in [ax.xaxis, ax.yaxis]:
        if axis.get_scale() == 'log':
            pass
        else:
            axis.set_minor_locator(ticker.AutoMinorLocator())

def _minor_ticks_linear(ax):
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())

def _grid(ax, log=False):
    ax.grid(True, which="major", lw=0.4, color="0.85", zorder=0)
    if not log:
        ax.grid(True, which="minor", lw=0.25, color="0.92", zorder=0)

def _savefig(fig, out):
    fig.tight_layout(pad=0.4)
    fig.savefig(out, format="pdf")
    print(f"[PLOT] -> {out}")
    plt.close(fig)

# ==============================================================================
# 2. PHYSICAL CONSTANTS & PARAMETERS
# ==============================================================================
c        = 299792458.0
lambda0  = 1550e-9
omega0   = 2 * np.pi * c / lambda0
n2       = 2.6e-20
A_eff    = 80e-12
gamma    = n2 * omega0 / (c * A_eff)
alpha_0  = 0.0
beta2_n  = +20e-27
beta2_a  = -20e-27
T0       = 10e-12
dz_base  = 0.5
T_window = 200 * T0
Nt       = 2**14
dt       = T_window / Nt
L_D      = T0**2 / abs(beta2_n)
P0_N1    = abs(beta2_a) / (gamma * T0**2)

sep       = "=" * 80
sep_s     = "-" * 80

print(sep)
print("  DM SOLITON SSFM -- v4.3 REFEREE-PROOF VALIDATION SUITE")
print(sep)
try:
    gpu_name = cp.cuda.runtime.getDeviceProperties(0)["name"].decode()
    mem_free, mem_total = cp.cuda.Device(0).mem_info
    print(f"\n[GPU] {gpu_name} | {mem_total/1024**3:.1f} GB total | {mem_free/1024**3:.2f} GB free")
except Exception as e:
    print(f"\n[GPU] Query failed: {e}")
print(f"\n  gamma={gamma:.3e} 1/(W*m)  |  L_D={L_D:.0f} m  |  P0_N1={P0_N1:.4f} W")
print(f"  Figures -> {os.path.abspath(FIG_DIR)}/")
print(sep_s)

# ==============================================================================
# 3. GPU GRIDS
# ==============================================================================
t_gpu   = cp.linspace(-T_window/2, T_window/2, Nt, dtype=cp.float32)
omega_g = (2*np.pi*cp.fft.fftfreq(Nt, dt)).astype(cp.float32)
t_64    = cp.linspace(-T_window/2, T_window/2, Nt, dtype=cp.float64)
w_64    = (2*np.pi*cp.fft.fftfreq(Nt, dt)).astype(cp.float64)

# ==============================================================================
# 4. SSFM OPERATORS -- float64 reference engine
# ==============================================================================
def _exp64(beta2, dz, alpha=0.0):
    hd = dz / 2
    return cp.exp((-0.5*alpha*hd) + (0.5j*beta2*w_64**2)*hd).astype(cp.complex128)

def _nl64(dz, gamma_val=None):
    g = gamma_val if gamma_val is not None else gamma
    return cp.complex128(1j * g * dz)

def _ssfm64(A, el, nl):
    A = cp.fft.ifft(cp.fft.fft(A) * el)
    A *= cp.exp(nl * (A * cp.conj(A)).real)
    A = cp.fft.ifft(cp.fft.fft(A) * el)
    return A

_Y_c1 = 1.0 / (2.0 - 2.0**(1.0/3.0))
_Y_c0 = 1.0 - 2.0*_Y_c1
_YOSHIDA_C = [_Y_c1, _Y_c0, _Y_c1]

def _ssfm64_ss4(A, beta2, dz, alpha=0.0):
    for c in _YOSHIDA_C:
        el = _exp64(beta2, c*dz, alpha)
        nl = _nl64(c*dz)
        A = _ssfm64(A, el, nl)
    return A

# ==============================================================================
# 5. PULSE FACTORIES
# ==============================================================================
def _pulse64(p0):
    return cp.sqrt(cp.float64(p0)) * cp.exp(-0.5*(t_64/T0)**2).astype(cp.complex128)

def _pulse64_sech(p0):
    return cp.sqrt(cp.float64(p0)) / cp.cosh(t_64/T0).astype(cp.complex128)

def _pulse64_two(p0, separation=6.0, phase_diff=np.pi):
    P0_f = float(p0)
    A1 = cp.sqrt(P0_f/2) * cp.exp(-0.5*((t_64 - separation*T0)/T0)**2)
    A2 = cp.sqrt(P0_f/2) * cp.exp(-0.5*((t_64 + separation*T0)/T0)**2) * cp.exp(1j*phase_diff)
    return (A1 + A2).astype(cp.complex128)

# ==============================================================================
# 6. INDEXING
# ==============================================================================
def build_index(Ltot, dz, L1, L2, method='traditional'):
    L_map = L1 + L2
    Nz    = int(np.round(Ltot/dz)) + 1
    k     = np.arange(1, Nz)
    z_eval = k*dz if method == 'traditional' else (k-0.5)*dz
    idx   = np.zeros(Nz, dtype=np.int8)
    idx[1:] = (z_eval % L_map >= L1).astype(np.int8)
    return Nz, idx

def make_indexer(method):
    return lambda Ltot, dz, L1, L2: build_index(Ltot, dz, L1, L2, method)

# ==============================================================================
# 7. SIMULATION RUNNERS
# ==============================================================================
def _energy(A):
    return float(cp.sum(cp.abs(A)**2) * dt)

def _l2rel(A, R):
    return float(cp.sqrt(cp.sum(cp.abs(A - R)**2) / cp.sum(cp.abs(R)**2)))

def _peak(A):
    return float(cp.max(cp.abs(A)**2))

def _fwhm(A):
    I = cp.abs(A)**2
    Im = cp.max(I)
    idx = cp.where(I > 0.5*Im)[0]
    return float((idx[-1] - idx[0]) * dt / T0) if len(idx) > 0 else 0.0

def run64(dz, indexer, Ltot, P0, L1, L2,
          pulse='gaussian', alpha=0.0, **kw):
    Nz, idx = indexer(Ltot, dz, L1, L2)
    en = _exp64(beta2_n, dz, alpha)
    ea = _exp64(beta2_a, dz, alpha)
    nl = _nl64(dz)
    A  = {'gaussian': _pulse64, 'sech': _pulse64_sech, 'two': _pulse64_two}[pulse](P0, **kw)
    for i in range(1, Nz):
        A = _ssfm64(A, ea if idx[i] else en, nl)
        if not cp.isfinite(cp.sum(A)):
            raise RuntimeError(f"NaN/Inf at step {i}, z={i*dz:.3f} m")
    return A

def run64_diag(dz, indexer, Ltot, P0, L1, L2,
               pulse='gaussian', alpha=0.0, rec=100, **kw):
    Nz, idx = indexer(Ltot, dz, L1, L2)
    en = _exp64(beta2_n, dz, alpha)
    ea = _exp64(beta2_a, dz, alpha)
    nl = _nl64(dz)
    A  = {'gaussian': _pulse64, 'sech': _pulse64_sech, 'two': _pulse64_two}[pulse](P0, **kw)
    E0 = _energy(A)
    z_r, e_r, p_r, f_r = [], [], [], []
    for i in range(1, Nz):
        A = _ssfm64(A, ea if idx[i] else en, nl)
        if not cp.isfinite(cp.sum(A)):
            raise RuntimeError(f"NaN/Inf at step {i}, z={i*dz:.3f} m")
        if i % rec == 0 or i == Nz-1:
            z_r.append(i*dz)
            e_r.append(_energy(A)/E0*100)
            p_r.append(_peak(A))
            f_r.append(_fwhm(A))
    return A, np.array(z_r), np.array(e_r), np.array(p_r), np.array(f_r)

def run64_ss4(dz, Ltot, P0, L1, L2, alpha=0.0):
    L_map = L1 + L2
    Nz    = int(np.round(Ltot/dz)) + 1
    A     = _pulse64(P0)
    for i in range(1, Nz):
        z_mid = (i-0.5)*dz
        beta2 = beta2_a if (z_mid % L_map >= L1) else beta2_n
        A = _ssfm64_ss4(A, beta2, dz, alpha)
        if not cp.isfinite(cp.sum(A)):
            raise RuntimeError(f"NaN/Inf at SS4 step {i}, z={i*dz:.3f} m")
    return A

# ==============================================================================
# 8. CONVERGENCE TEST ENGINE
# ==============================================================================
def convergence_test(Ltot, L1, L2, P0, dz_list, ref_dz,
                     pulse='gaussian', alpha=0.0, label="Case", **kw):
    print(f"\n[{label}]  Ltot={Ltot:.0f} m  L1={L1:.0f}  L2={L2:.0f}  P0={P0:.2f} W  alpha={alpha:.2e}")
    t0 = time.time()
    A_ref = run64(ref_dz, make_indexer('midpoint'), Ltot, P0, L1, L2, pulse, alpha, **kw)
    print(f"  Ref (dz={ref_dz:.5f} m) built in {time.time()-t0:.1f} s")

    et, em, tt, tm = [], [], [], []
    for dh in tqdm(dz_list, ncols=72, desc=label):
        t0 = time.time()
        At = run64(dh, make_indexer('traditional'), Ltot, P0, L1, L2, pulse, alpha, **kw)
        tt.append(time.time() - t0)
        t0 = time.time()
        Am = run64(dh, make_indexer('midpoint'),    Ltot, P0, L1, L2, pulse, alpha, **kw)
        tm.append(time.time() - t0)
        et.append(_l2rel(At, A_ref))
        em.append(_l2rel(Am, A_ref))

    ot = [np.nan] + [np.log(et[i-1]/et[i])/np.log(dz_list[i-1]/dz_list[i])
                     for i in range(1, len(dz_list))]
    om = [np.nan] + [np.log(em[i-1]/em[i])/np.log(dz_list[i-1]/dz_list[i])
                     for i in range(1, len(dz_list))]
    avg_ot = np.nanmean(ot)
    avg_om = np.nanmean(om)

    print(f"\n  {'dz':>6} | {'Err Trad':>10} | {'p_T':>6} | {'Err Mid':>10} | {'p_M':>6} | {'Spdup':>6}")
    print(f"  {'-'*6}-|-{'-'*10}-|-{'-'*6}-|-{'-'*10}-|-{'-'*6}-|-{'-'*6}")
    for dh, e_t, o_t, e_m, o_m, t_t, t_m in zip(dz_list, et, ot, em, om, tt, tm):
        print(f"  {dh:6.3f} | {e_t:10.2e} | {o_t:6.3f} | {e_m:10.2e} | {o_m:6.3f} | {t_t/t_m:6.2f}")
    print(f"  => Trad avg order: {avg_ot:.3f}  |  Mid avg order: {avg_om:.3f}")
    if len(dz_list) > 1 and em[1] > 0:
        print(f"  => Error reduction @ dz={dz_list[1]}: {et[1]/em[1]:.0f}x")

    return dict(dz_list=dz_list, et=et, em=em, ot=ot, om=om,
                tt=tt, tm=tm, avg_ot=avg_ot, avg_om=avg_om,
                A_ref=A_ref, label=label)

# ==============================================================================
# 9. PLOTTING LIBRARY -- Individual plots (kept for standalone use)
# ==============================================================================

def _fit_slope(x, y, n=None):
    n = n or max(len(x)-1, 2)
    s, b, r, *_ = linregress(np.log10(x[:n]), np.log10(y[:n]))
    return s, b, r**2

def plot_convergence(data, fname, title=None, show_ss4=None):
    dz  = np.array(data['dz_list'])
    et  = np.array(data['et'])
    em  = np.array(data['em'])
    st, bt, _ = _fit_slope(dz, et)
    sm, bm, _ = _fit_slope(dz, em)
    dz_f = np.logspace(np.log10(dz.min()*0.65), np.log10(dz.max()*1.5), 120)
    anchor = em[0] / dz[0]**2
    ref1 = anchor * dz_f**1
    ref2 = anchor * dz_f**2

    fig, ax = plt.subplots(figsize=(W1, H1*1.1))
    ax.loglog(dz, et, "s", c=C_TRAD, mfc="w", mec=C_TRAD, ms=5, mew=0.8,
              label="Traditional", zorder=5, clip_on=False)
    ax.loglog(dz_f, 10**bt*dz_f**st, "--", c=C_TRAD, lw=1.0,
              label=rf"Fit $p={st:.2f}$", zorder=4)
    ax.loglog(dz, em, "o", c=C_MID, mfc="w", mec=C_MID, ms=5, mew=0.8,
              label="Midpoint (this work)", zorder=5, clip_on=False)
    ax.loglog(dz_f, 10**bm*dz_f**sm, "-", c=C_MID, lw=1.0,
              label=rf"Fit $p={sm:.2f}$", zorder=4)
    if show_ss4 is not None:
        dz4 = np.array(show_ss4['dz_list'])
        e4  = np.array(show_ss4['errs'])
        s4, b4, _ = _fit_slope(dz4, e4)
        ref4 = (e4[0]/dz4[0]**4) * dz_f**4
        ax.loglog(dz4, e4, "D", c=C_SS4, mfc="w", mec=C_SS4, ms=5, mew=0.8,
                  label="SS4 (Yoshida)", zorder=5, clip_on=False)
        ax.loglog(dz_f, 10**b4*dz_f**s4, "-.", c=C_SS4, lw=1.0,
                  label=rf"Fit $p={s4:.2f}$", zorder=4)
        ax.loglog(dz_f, ref4, ":", c=C_REF4, lw=0.7,
                  label=r"$\mathcal{O}(\Delta z^4)$", zorder=2)
    ax.loglog(dz_f, ref1, ":", c=C_REF1, lw=0.7,
              label=r"$\mathcal{O}(\Delta z)$", zorder=2)
    ax.loglog(dz_f, ref2, ":", c=C_REF2, lw=0.7,
              label=r"$\mathcal{O}(\Delta z^2)$", zorder=2)
    ax.set_xlabel(r"Step size $\Delta z$ (m)")
    ax.set_ylabel(r"Relative $L_2$ error")
    if title:
        ax.set_title(title, pad=3)
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", ncol=2, handlelength=1.4)
    _grid(ax, log=True)
    _savefig(fig, figpath(fname))


def plot_local_error_both(z_trad, err_trad, z_mid, err_mid, z_interfaces, fname):
    fig, ax = plt.subplots(figsize=(W1, H1))
    ax.semilogy(z_trad, err_trad, c=C_TRAD, lw=1.0, label="Traditional")
    ax.semilogy(z_mid,  err_mid,  c=C_MID,  lw=1.0, label="Midpoint (this work)")
    for i, zif in enumerate(z_interfaces):
        ax.axvline(zif, c="0.5", lw=0.7, ls="--",
                   label=r"$\beta_2$ interface" if i == 0 else "_")
    ax.set_xlabel(r"Propagation distance $z$ (m)")
    ax.set_ylabel(r"Cumulative $L_2$ error")
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8")
    _minor_ticks_linear(ax)
    _grid(ax)
    _savefig(fig, figpath(fname))


def plot_order_vs_ratio(ratios, orders_trad, orders_mid, fname):
    fig, ax = plt.subplots(figsize=(W1, H1))
    ax.semilogx(ratios, orders_trad, "s-", c=C_TRAD, ms=5, mfc="w", mec=C_TRAD,
                lw=1.0, label="Traditional")
    ax.semilogx(ratios, orders_mid,  "o-", c=C_MID,  ms=5, mfc="w", mec=C_MID,
                lw=1.0, label="Midpoint (this work)")
    ax.axhline(1.0, c=C_REF1, lw=0.7, ls=":", label=r"$p=1$ (1st-order)")
    ax.axhline(2.0, c=C_REF2, lw=0.7, ls=":", label=r"$p=2$ (2nd-order)")
    ax.set_xlabel(r"$L_\mathrm{map}/\Delta z$ (steps per map period)")
    ax.set_ylabel("Observed convergence order $p$")
    ax.set_ylim(-0.5, 2.5)
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8")
    _minor_ticks_linear(ax)
    _grid(ax)
    _savefig(fig, figpath(fname))


def plot_ss4_comparison(data_mid, data_ss4, fname):
    fig, ax = plt.subplots(figsize=(W1, H1*1.1))
    ax.loglog(data_mid['tt'], data_mid['et'], "s--", c=C_TRAD, ms=5,
              mfc="w", mec=C_TRAD, lw=1.0, label="Trad (2nd-order SSFM)", alpha=0.7)
    ax.loglog(data_mid['tm'], data_mid['em'], "o-",  c=C_MID,  ms=5,
              mfc="w", mec=C_MID,  lw=1.0, label="Midpoint (this work, 2nd-order)")
    ax.loglog(data_ss4['times'], data_ss4['errs'], "D-.", c=C_SS4, ms=5,
              mfc="w", mec=C_SS4,  lw=1.0, label="SS4/Yoshida (4th-order)")
    ax.set_xlabel("Wall-clock time (s)")
    ax.set_ylabel(r"Relative $L_2$ error")
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", handlelength=1.4)
    _grid(ax, log=True)
    _savefig(fig, figpath(fname))


def plot_efficiency_pareto(all_data, fname):
    colors = [C_TRAD, C_MID, C_SS4, C_REF4, "#E69F00", "#56B4E9"]
    fig, ax = plt.subplots(figsize=(W1, H1*1.1))
    for k, d in enumerate(all_data):
        c = colors[k % len(colors)]
        lbl = d.get('label', f"Case {k+1}")
        ax.loglog(d['tt'], d['et'], "s--", c=c, ms=4, lw=0.8,
                  alpha=0.55, mfc="w", mec=c)
        ax.loglog(d['tm'], d['em'], "o-",  c=c, ms=4, lw=0.8,
                  mfc="w", mec=c, label=lbl)
    from matplotlib.lines import Line2D
    h, l = ax.get_legend_handles_labels()
    extras = [
        Line2D([0],[0], ls="--", c="0.4", lw=0.8, marker="s", ms=4, mfc="w",
               label="Traditional"),
        Line2D([0],[0], ls="-",  c="0.4", lw=0.8, marker="o", ms=4, mfc="w",
               label="Midpoint"),
    ]
    ax.legend(handles=h+extras, frameon=True, framealpha=0.9, edgecolor="0.8",
              ncol=2, handlelength=1.4)
    ax.set_xlabel("Wall-clock time (s)")
    ax.set_ylabel(r"Relative $L_2$ error")
    _grid(ax, log=True)
    _savefig(fig, figpath(fname))


# ==============================================================================
# 9b. PANEL PLOTS -- Merged multi-subfigure composites, NO empty subplots
# ==============================================================================

def _add_panel_label(ax, label, x=-0.18, y=1.05):
    """Add (a), (b), (c) etc. panel labels in bold."""
    ax.text(x, y, f"({label})", transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='top', ha='right')


def plot_panel_case1(data, zr, etr, emr, ptr, zr2, pmr, fname):
    """Case 1: 1x3 panel -- convergence | energy | peak power."""
    dz  = np.array(data['dz_list'])
    et  = np.array(data['et'])
    em  = np.array(data['em'])
    st, bt, _ = _fit_slope(dz, et)
    sm, bm, _ = _fit_slope(dz, em)
    dz_f = np.logspace(np.log10(dz.min()*0.65), np.log10(dz.max()*1.5), 120)
    anchor = em[0] / dz[0]**2
    ref1 = anchor * dz_f**1
    ref2 = anchor * dz_f**2

    fig, axes = plt.subplots(1, 3, figsize=(W2, W2/3.2))

    ax = axes[0]
    ax.loglog(dz, et, "s", c=C_TRAD, mfc="w", mec=C_TRAD, ms=5, mew=0.8,
              label="Traditional", zorder=5, clip_on=False)
    ax.loglog(dz_f, 10**bt*dz_f**st, "--", c=C_TRAD, lw=1.0,
              label=rf"Fit $p={st:.2f}$", zorder=4)
    ax.loglog(dz, em, "o", c=C_MID, mfc="w", mec=C_MID, ms=5, mew=0.8,
              label="Midpoint (this work)", zorder=5, clip_on=False)
    ax.loglog(dz_f, 10**bm*dz_f**sm, "-", c=C_MID, lw=1.0,
              label=rf"Fit $p={sm:.2f}$", zorder=4)
    ax.loglog(dz_f, ref1, ":", c=C_REF1, lw=0.7, label=r"$\mathcal{O}(\Delta z)$", zorder=2)
    ax.loglog(dz_f, ref2, ":", c=C_REF2, lw=0.7, label=r"$\mathcal{O}(\Delta z^2)$", zorder=2)
    ax.set_xlabel(r"Step size $\Delta z$ (m)")
    ax.set_ylabel(r"Relative $L_2$ error")
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", ncol=1, handlelength=1.4, fontsize=6.5)
    _grid(ax, log=True)
    _add_panel_label(ax, "a")

    ax = axes[1]
    ax.plot(zr, etr, c=C_TRAD, lw=1.0, label="Traditional")
    ax.plot(zr, emr, c=C_MID,  lw=1.0, label="Midpoint")
    ax.axhline(100, c="0.5", lw=0.6, ls=":")
    ax.set_xlabel(r"$z$ (m)")
    ax.set_ylabel("Energy retention (%)")
    ax.set_ylim(99.9, 100.1)
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", fontsize=6.5)
    _minor_ticks_linear(ax)
    _grid(ax)
    _add_panel_label(ax, "b")

    ax = axes[2]
    ax.plot(zr, ptr, c=C_TRAD, lw=1.0, label="Traditional")
    ax.plot(zr2, pmr, c=C_MID,  lw=1.0, label="Midpoint")
    ax.set_xlabel(r"$z$ (m)")
    ax.set_ylabel("Peak power (W)")
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", fontsize=6.5)
    _minor_ticks_linear(ax)
    _grid(ax)
    _add_panel_label(ax, "c")

    fig.tight_layout(pad=0.5)
    fig.savefig(figpath(fname), format="pdf")
    print(f"[PANEL] -> {figpath(fname)}")
    plt.close(fig)


def plot_panel_case7(zr, etr, emr, ptr, zr2, pmr, ftr, fmr, fname):
    """Case 7: 1x3 panel -- energy | peak power | FWHM over 50 km."""
    fig, axes = plt.subplots(1, 3, figsize=(W2, W2/3.2))

    ax = axes[0]
    ax.plot(zr/1000, etr, c=C_TRAD, lw=1.0, label="Traditional")
    ax.plot(zr/1000, emr, c=C_MID,  lw=1.0, label="Midpoint")
    ax.axhline(100, c="0.5", lw=0.6, ls=":")
    ax.set_xlabel(r"$z$ (km)")
    ax.set_ylabel("Energy retention (%)")
    ax.set_ylim(99.5, 100.5)
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", fontsize=6.5)
    _minor_ticks_linear(ax)
    _grid(ax)
    _add_panel_label(ax, "a")

    ax = axes[1]
    ax.plot(zr/1000, ptr, c=C_TRAD, lw=1.0, label="Traditional")
    ax.plot(zr2/1000, pmr, c=C_MID,  lw=1.0, label="Midpoint")
    ax.set_xlabel(r"$z$ (km)")
    ax.set_ylabel("Peak power (W)")
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", fontsize=6.5)
    _minor_ticks_linear(ax)
    _grid(ax)
    _add_panel_label(ax, "b")

    ax = axes[2]
    ax.plot(zr/1000,  ftr, c=C_TRAD, lw=1.0, label="Traditional")
    ax.plot(zr2/1000, fmr, c=C_MID,  lw=1.0, label="Midpoint")
    ax.set_xlabel(r"$z$ (km)")
    ax.set_ylabel(r"FWHM / $T_0$")
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", fontsize=6.5)
    _minor_ticks_linear(ax)
    _grid(ax)
    _add_panel_label(ax, "c")

    fig.tight_layout(pad=0.5)
    fig.savefig(figpath(fname), format="pdf")
    print(f"[PANEL] -> {figpath(fname)}")
    plt.close(fig)


def plot_panel_cases234(data_list, labels, titles, fname):
    """
    Cases 2,3,4: 1x3 panel -- three convergence curves side by side.
    data_list: list of 3 convergence dicts.
    """
    fig, axes = plt.subplots(1, 3, figsize=(W2, W2/3.2))
    for idx, (data, lbl, ttl) in enumerate(zip(data_list, labels, titles)):
        dz = np.array(data['dz_list'])
        et = np.array(data['et'])
        em = np.array(data['em'])
        st, bt, _ = _fit_slope(dz, et)
        sm, bm, _ = _fit_slope(dz, em)
        dz_f = np.logspace(np.log10(dz.min()*0.65), np.log10(dz.max()*1.5), 120)
        anchor = em[0] / dz[0]**2
        ref1 = anchor * dz_f**1
        ref2 = anchor * dz_f**2

        ax = axes[idx]
        ax.loglog(dz, et, "s", c=C_TRAD, mfc="w", mec=C_TRAD, ms=5, mew=0.8,
                  label="Traditional", zorder=5, clip_on=False)
        ax.loglog(dz_f, 10**bt*dz_f**st, "--", c=C_TRAD, lw=1.0, zorder=4)
        ax.loglog(dz, em, "o", c=C_MID, mfc="w", mec=C_MID, ms=5, mew=0.8,
                  label="Midpoint", zorder=5, clip_on=False)
        ax.loglog(dz_f, 10**bm*dz_f**sm, "-", c=C_MID, lw=1.0, zorder=4)
        ax.loglog(dz_f, ref1, ":", c=C_REF1, lw=0.7, zorder=2)
        ax.loglog(dz_f, ref2, ":", c=C_REF2, lw=0.7, zorder=2)
        ax.set_xlabel(r"Step size $\Delta z$ (m)")
        ax.set_ylabel(r"Relative $L_2$ error")
        ax.set_title(ttl, pad=3, fontsize=8)
        if idx == 2:
            ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", ncol=1,
                      handlelength=1.4, fontsize=6.5)
        _grid(ax, log=True)
        _add_panel_label(ax, chr(ord('a')+idx))

    fig.tight_layout(pad=0.5)
    fig.savefig(figpath(fname), format="pdf")
    print(f"[PANEL] -> {figpath(fname)}")
    plt.close(fig)


def plot_panel_cases1011(data10, data11, fname):
    """Cases 10,11: 1x2 panel -- sech IC | lossy fiber convergence."""
    fig, axes = plt.subplots(1, 2, figsize=(W2*0.67, W2*0.67/3.2))
    for idx, (data, ttl) in enumerate([(data10, "Sech pulse"), (data11, r"Lossy fiber, $\alpha=0.2$ dB/km")]):
        dz = np.array(data['dz_list'])
        et = np.array(data['et'])
        em = np.array(data['em'])
        st, bt, _ = _fit_slope(dz, et)
        sm, bm, _ = _fit_slope(dz, em)
        dz_f = np.logspace(np.log10(dz.min()*0.65), np.log10(dz.max()*1.5), 120)
        anchor = em[0] / dz[0]**2
        ref1 = anchor * dz_f**1
        ref2 = anchor * dz_f**2

        ax = axes[idx]
        ax.loglog(dz, et, "s", c=C_TRAD, mfc="w", mec=C_TRAD, ms=5, mew=0.8,
                  label="Traditional", zorder=5, clip_on=False)
        ax.loglog(dz_f, 10**bt*dz_f**st, "--", c=C_TRAD, lw=1.0, zorder=4)
        ax.loglog(dz, em, "o", c=C_MID, mfc="w", mec=C_MID, ms=5, mew=0.8,
                  label="Midpoint", zorder=5, clip_on=False)
        ax.loglog(dz_f, 10**bm*dz_f**sm, "-", c=C_MID, lw=1.0, zorder=4)
        ax.loglog(dz_f, ref1, ":", c=C_REF1, lw=0.7, zorder=2)
        ax.loglog(dz_f, ref2, ":", c=C_REF2, lw=0.7, zorder=2)
        ax.set_xlabel(r"Step size $\Delta z$ (m)")
        ax.set_ylabel(r"Relative $L_2$ error")
        ax.set_title(ttl, pad=3, fontsize=8)
        if idx == 1:
            ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", ncol=1,
                      handlelength=1.4, fontsize=6.5)
        _grid(ax, log=True)
        _add_panel_label(ax, chr(ord('a')+idx))

    fig.tight_layout(pad=0.5)
    fig.savefig(figpath(fname), format="pdf")
    print(f"[PANEL] -> {figpath(fname)}")
    plt.close(fig)


# ==============================================================================
# 10. CASE RUNNERS
# ==============================================================================
results_store = {}

# ------------------------------------------------------------------------------
# CASE 1 -- Baseline symmetric map (1x3 panel)
# ------------------------------------------------------------------------------
def run_case_1():
    print(sep)
    print("CASE 1: BASELINE SYMMETRIC MAP  N≈1.4")
    print(sep_s)
    L1 = L2 = 50.0
    Ltot = 6000.0
    P0 = 0.30
    dz_list = [2.0, 1.0, 0.5, 0.25]
    ref_dz = dz_base / 32
    d = convergence_test(Ltot, L1, L2, P0, dz_list, ref_dz, label="Case 1 (Baseline)")
    results_store['case1'] = d

    dz_t = 1.0
    _, zr, etr, ptr, _ = run64_diag(dz_t, make_indexer('traditional'), Ltot, P0, L1, L2, rec=100)
    _, zr2, emr, pmr, _ = run64_diag(dz_t, make_indexer('midpoint'), Ltot, P0, L1, L2, rec=100)

    plot_panel_case1(d, zr, etr, emr, ptr, zr2, pmr, "Fig01_Case1_Baseline_Panel.pdf")
    print(sep)

# ------------------------------------------------------------------------------
# CASE 2 -- Strong breathing
# ------------------------------------------------------------------------------
def run_case_2():
    print(sep)
    print("CASE 2: STRONG BREATHING  P0=0.60 W  N≈2.0")
    print(sep_s)
    L1 = L2 = 50.0
    Ltot = 2000.0
    P0 = 0.60
    dz_list = [1.0, 0.5, 0.25]
    ref_dz = dz_base / 16
    d = convergence_test(Ltot, L1, L2, P0, dz_list, ref_dz, label="Case 2 (Strong)")
    results_store['case2'] = d
    print(sep)

# ------------------------------------------------------------------------------
# CASE 3 -- Asymmetric map 60:40
# ------------------------------------------------------------------------------
def run_case_3():
    print(sep)
    print("CASE 3: ASYMMETRIC MAP  60:40 DUTY CYCLE")
    print(sep_s)
    L1, L2 = 60.0, 40.0
    Ltot = 2000.0
    P0 = 0.30
    dz_list = [1.0, 0.5, 0.25]
    ref_dz = dz_base / 16
    d = convergence_test(Ltot, L1, L2, P0, dz_list, ref_dz, label="Case 3 (60:40)")
    results_store['case3'] = d
    print(sep)

# ------------------------------------------------------------------------------
# CASE 4 -- Two-soliton collision
# ------------------------------------------------------------------------------
def run_case_4():
    print(sep)
    print("CASE 4: TWO-SOLITON COLLISION  phase=π  sep=6T0")
    print(sep_s)
    L1 = L2 = 50.0
    Ltot = 2000.0
    P0 = 0.30
    dz_list = [1.0, 0.5, 0.25]
    ref_dz = dz_base / 16
    d = convergence_test(Ltot, L1, L2, P0, dz_list, ref_dz,
                         pulse='two', label="Case 4 (Two-soliton)",
                         separation=6.0, phase_diff=np.pi)
    results_store['case4'] = d
    print(sep)

# ------------------------------------------------------------------------------
# CASE 5 -- Local error microscope
# ------------------------------------------------------------------------------
def run_case_5():
    print(sep)
    print("CASE 5 (EXTENDED): LOCAL ERROR MICROSCOPE -- Both Methods, 3 Map Periods")
    print(sep_s)
    L1 = L2 = 50.0
    Ltot = 300.0
    P0 = 0.30
    dz_test = 0.5
    dz_ref = 0.005

    ratio = int(np.round(dz_test / dz_ref))
    Nz_ref = int(np.round(Ltot / dz_ref)) + 1
    nl_r = _nl64(dz_ref)
    A_r = _pulse64(P0)
    snaps, z_snaps = [], []
    for i in range(1, Nz_ref):
        z_mid_ref = (i - 0.5) * dz_ref
        b2 = beta2_a if (z_mid_ref % (L1 + L2) >= L1) else beta2_n
        A_r = _ssfm64(A_r, _exp64(b2, dz_ref), nl_r)
        if i % ratio == 0:
            snaps.append(A_r.copy())
            z_snaps.append(i * dz_ref)

    local = {}
    for mname, meth in [("Traditional", 'traditional'), ("Midpoint", 'midpoint')]:
        Nz, idx = build_index(Ltot, dz_test, L1, L2, meth)
        en = _exp64(beta2_n, dz_test)
        ea = _exp64(beta2_a, dz_test)
        nl = _nl64(dz_test)
        A = _pulse64(P0)
        errs, zs = [], []
        for i in range(1, Nz):
            A = _ssfm64(A, ea if idx[i] else en, nl)
            si = i - 1
            if si < len(snaps):
                errs.append(_l2rel(A, snaps[si]))
                zs.append(i * dz_test)
        local[mname] = (np.array(zs), np.array(errs))
        print(f"  [{mname}] max={max(errs):.2e}  mean={np.mean(errs):.2e}  "
              f"ratio(max/mean)={max(errs)/np.mean(errs):.2f}")

    z_iface = [L1, L1+L2, 2*L1+L2, 2*(L1+L2), 2*L1+2*L2]
    z_iface = [z for z in z_iface if z <= Ltot]
    plot_local_error_both(
        local["Traditional"][0], local["Traditional"][1],
        local["Midpoint"][0],    local["Midpoint"][1],
        z_iface,
        "Fig05_Case5_LocalError_BothMethods.pdf"
    )
    print(sep)

# ------------------------------------------------------------------------------
# CASE 6 -- Efficiency Pareto (10 km)
# ------------------------------------------------------------------------------
def run_case_6():
    print(sep)
    print("CASE 6: EFFICIENCY PARETO  L=10 km")
    print(sep_s)
    L1 = L2 = 50.0
    Ltot = 10000.0
    P0 = 0.30
    dz_list = [2.0, 1.0, 0.5, 0.25, 0.125]
    ref_dz = dz_base / 8
    d = convergence_test(Ltot, L1, L2, P0, dz_list, ref_dz, label="Case 6 (Efficiency)")
    results_store['case6'] = d
    print(sep)

# ------------------------------------------------------------------------------
# CASE 7 -- Long-haul 50 km (1x3 panel)
# ------------------------------------------------------------------------------
def run_case_7():
    print(sep)
    print("CASE 7 (EXTENDED): LONG-HAUL 50 km -- Energy + FWHM + Peak drift")
    print(sep_s)
    L1 = L2 = 50.0
    Ltot = 50000.0
    P0 = 0.30
    dz = 1.0

    _, zr, etr, ptr, ftr = run64_diag(dz, make_indexer('traditional'), Ltot, P0, L1, L2, rec=500)
    _, zr2, emr, pmr, fmr = run64_diag(dz, make_indexer('midpoint'), Ltot, P0, L1, L2, rec=500)

    print(f"  Traditional: E_final={etr[-1]:.6f}%  peak_drift={(ptr[-1]-ptr[0])/ptr[0]*100:.3f}%")
    print(f"  Midpoint:    E_final={emr[-1]:.6f}%  peak_drift={(pmr[-1]-pmr[0])/pmr[0]*100:.3f}%")
    print(f"  Peak-drift difference = {abs((pmr[-1]-pmr[0])/pmr[0]-(ptr[-1]-ptr[0])/ptr[0])*100:.3f} pp")

    plot_panel_case7(zr, etr, emr, ptr, zr2, pmr, ftr, fmr,
                     "Fig07_Case7_LongHaul_Panel.pdf")
    print(sep)

# ------------------------------------------------------------------------------
# CASE 8 -- Step-ratio robustness
# ------------------------------------------------------------------------------
def run_case_8():
    print(sep)
    print("CASE 8 (NEW): STEP-RATIO ROBUSTNESS -- order vs L_map/dz")
    print(sep_s)
    L1 = L2 = 50.0
    L_map = L1 + L2
    Ltot = 2000.0
    P0 = 0.30

    ratios_target = [10, 20, 40, 80, 160, 320, 640]
    ratios_actual, orders_trad, orders_mid = [], [], []

    for r in tqdm(ratios_target, ncols=72, desc="Case 8"):
        dz_coarse = L_map / r
        dz_fine = dz_coarse / 2
        dz_finer = dz_fine / 2
        ref_dz = dz_finer / 8

        A_ref = run64(ref_dz, make_indexer('midpoint'), Ltot, P0, L1, L2)

        errors = {}
        for meth in ('traditional', 'midpoint'):
            ix = make_indexer(meth)
            e = [_l2rel(run64(dz, ix, Ltot, P0, L1, L2), A_ref)
                 for dz in (dz_coarse, dz_fine, dz_finer)]
            errors[meth] = e

        ot = np.mean([np.log(errors['traditional'][i-1] / errors['traditional'][i]) /
                      np.log(2) for i in range(1, 3)])
        om = np.mean([np.log(errors['midpoint'][i-1] / errors['midpoint'][i]) /
                      np.log(2) for i in range(1, 3)])
        ratios_actual.append(r)
        orders_trad.append(ot)
        orders_mid.append(om)
        note = "  <-- float64 noise floor" if om < 1.0 else ""
        print(f"  ratio={r:4d}  dz_coarse={dz_coarse:.4f} m  "
              f"order_trad={ot:.3f}  order_mid={om:.3f}{note}")

    plot_order_vs_ratio(ratios_actual, orders_trad, orders_mid,
                        "Fig08_Case8_StepRatioRobustness.pdf")
    results_store['case8'] = dict(ratios=ratios_actual,
                                  orders_trad=orders_trad,
                                  orders_mid=orders_mid)
    print(sep)

# ------------------------------------------------------------------------------
# CASE 9 -- SS4 Comparison
# ------------------------------------------------------------------------------
def run_case_9():
    print(sep)
    print("CASE 9 (NEW): HIGHER-ORDER COMPARISON -- Midpoint-2nd vs SS4 (Yoshida)")
    print(sep_s)
    L1 = L2 = 50.0
    Ltot = 2000.0
    P0 = 0.30

    dz_list_23 = [2.0, 1.0, 0.5, 0.25, 0.125]
    dz_list_ss4 = [4.0, 2.0, 1.0, 0.5, 0.25]
    ref_dz = dz_base / 16

    A_ref = run64(ref_dz, make_indexer('midpoint'), Ltot, P0, L1, L2)

    et, em, tt, tm = [], [], [], []
    for dh in tqdm(dz_list_23, ncols=72, desc="Case 9 (2nd-order)"):
        t0 = time.time()
        At = run64(dh, make_indexer('traditional'), Ltot, P0, L1, L2)
        tt.append(time.time() - t0)
        t0 = time.time()
        Am = run64(dh, make_indexer('midpoint'), Ltot, P0, L1, L2)
        tm.append(time.time() - t0)
        et.append(_l2rel(At, A_ref))
        em.append(_l2rel(Am, A_ref))

    times_ss4, errs_ss4 = [], []
    for dh in tqdm(dz_list_ss4, ncols=72, desc="Case 9 (SS4)"):
        t0 = time.time()
        A4 = run64_ss4(dh, Ltot, P0, L1, L2)
        times_ss4.append(time.time() - t0)
        errs_ss4.append(_l2rel(A4, A_ref))

    print(f"\n  2nd-order results:")
    for dh, e_t, e_m, t_t, t_m in zip(dz_list_23, et, em, tt, tm):
        print(f"    dz={dh:.3f}  Trad={e_t:.2e} ({t_t:.2f}s)  "
              f"Mid={e_m:.2e} ({t_m:.2f}s)  ratio={e_t/e_m:.0f}x")
    print(f"\n  SS4 results:")
    for dh, e4, t4 in zip(dz_list_ss4, errs_ss4, times_ss4):
        print(f"    dz={dh:.3f}  SS4={e4:.2e} ({t4:.2f}s)  SS4_cost/Mid_cost≈3x")

    data2nd = dict(tt=tt, tm=tm, et=et, em=em, label="2nd-order SSFM")
    data_ss4_plot = dict(times=times_ss4, errs=errs_ss4)
    plot_ss4_comparison(data2nd, data_ss4_plot,
                        "Fig09_Case9_SS4_Comparison.pdf")
    results_store['case9_2nd'] = data2nd
    print(sep)

# ------------------------------------------------------------------------------
# CASE 10 -- Initial condition generalization: sech pulse
# ------------------------------------------------------------------------------
def run_case_10():
    print(sep)
    print("CASE 10 (NEW): INITIAL CONDITION -- sech pulse (non-Gaussian)")
    print(sep_s)
    L1 = L2 = 50.0
    Ltot = 2000.0
    P0 = 0.30
    dz_list = [1.0, 0.5, 0.25]
    ref_dz = dz_base / 16
    d = convergence_test(Ltot, L1, L2, P0, dz_list, ref_dz,
                         pulse='sech', label="Case 10 (sech IC)")
    results_store['case10'] = d
    print(sep)

# ------------------------------------------------------------------------------
# CASE 11 -- Loss/gain fiber (alpha ≠ 0)
# ------------------------------------------------------------------------------
def run_case_11():
    print(sep)
    print("CASE 11 (NEW): LOSSY FIBER  alpha=0.2 dB/km")
    print(sep_s)
    alpha_dBkm = 0.2
    alpha_pm = alpha_dBkm * np.log(10) / (10 * 1000)
    print(f"  alpha = {alpha_dBkm} dB/km = {alpha_pm:.4e} Np/m")
    L1 = L2 = 50.0
    Ltot = 2000.0
    P0 = 0.30
    dz_list = [1.0, 0.5, 0.25]
    ref_dz = dz_base / 16
    d = convergence_test(Ltot, L1, L2, P0, dz_list, ref_dz,
                         alpha=alpha_pm, label="Case 11 (Lossy)")
    results_store['case11'] = d
    print(sep)

# ==============================================================================
# 11. MAIN
# ==============================================================================
if __name__ == "__main__":
    run_case_1()
    run_case_2()
    run_case_3()
    run_case_4()
    run_case_5()
    run_case_6()
    run_case_7()
    run_case_8()
    run_case_9()
    run_case_10()
    run_case_11()

    # ---- MERGED PANELS (post-run, after all data collected) ----
    print(sep)
    print("MERGED PANEL GENERATION")
    print(sep_s)

    # Cases 2,3,4 -> 1x3 panel
    plot_panel_cases234(
        [results_store['case2'], results_store['case3'], results_store['case4']],
        ["Strong", "Asymmetric 60:40", "Two-soliton"],
        [r"$P_0=0.60$ W, $N\approx2.0$",
         r"Asymmetric map $60\!:\!40$",
         r"Two-soliton collision, $\Delta\phi=\pi$"],
        "Fig02_Cases234_Convergence_Panel.pdf"
    )

    # Cases 10,11 -> 1x2 panel
    plot_panel_cases1011(
        results_store['case10'], results_store['case11'],
        "Fig10_Cases1011_Generalization_Panel.pdf"
    )

    # Global Pareto
    print(sep)
    print("GLOBAL EFFICIENCY PARETO")
    plot_efficiency_pareto(
        [results_store['case1'], results_store['case2'],
         results_store['case3'], results_store['case4']],
        "Fig11_Global_Pareto.pdf"
    )

    # ---- Summary ----
    print("\n" + sep)
    print("✅  v4.3 REFEREE-PROOF VALIDATION COMPLETE")
    print(f"\n  All {len([f for f in os.listdir(FIG_DIR) if f.endswith('.pdf')])} "
          f"figures (PDF, vector) in: {os.path.abspath(FIG_DIR)}/\n")

    hdr = f"  {'Case':>22} | {'p_Trad':>7} | {'p_Mid':>7} | {'Referee question answered'}"
    print(hdr)
    print("  " + "-" * (len(hdr)-2))
    rows = [
        ("1 Baseline",         'case1',  "N≈1.4 breathing -- baseline"),
        ("2 Strong breathing",  'case2',  "Non-integrable dynamics"),
        ("3 Asymmetric 60:40",  'case3',  "Map symmetry independence"),
        ("4 Two-soliton",       'case4',  "Nonlinear interaction"),
        ("6 Efficiency",        'case6',  "Engineering value"),
        ("10 Sech IC",          'case10', "Non-Gaussian initial condition"),
        ("11 Lossy fiber",      'case11', "Loss/gain (alpha≠0)"),
    ]
    for name, key, msg in rows:
        d = results_store[key]
        print(f"  {name:>22} | {d['avg_ot']:7.3f} | {d['avg_om']:7.3f} | {msg}")

    print("\n  STEP-RATIO ROBUSTNESS (Case 8):")
    d8 = results_store['case8']
    for r, ot, om in zip(d8['ratios'], d8['orders_trad'], d8['orders_mid']):
        flag_t = "⚠" if ot > 1.5 else "✓"
        flag_m = "✓" if om > 1.7 else "⚠"
        note = " (float64 floor)" if om < 1.0 else ""
        print(f"    L_map/dz={r:4d}  p_trad={ot:.3f} {flag_t}  p_mid={om:.3f} {flag_m}{note}")

    print("\n  REFEREE ANSWER CHECKLIST:")
    checks = [
        ("Q1", "N=1 only?",          "Cases 2,4 -- N≈2 & collision"),
        ("Q2", "Symmetry needed?",    "Case 3 -- 60:40 asymmetric map"),
        ("Q3", "Physical mechanism?", "Case 5 -- both-method local error, spike at interface"),
        ("Q4", "Computational value?", "Case 6 Pareto; Case 9 SS4 comparison"),
        ("Q5", "Long distance OK?",   "Case 7 -- 50 km, energy+FWHM+peak"),
        ("Q6", "dz-ratio dependent?", "Case 8 -- L_map/dz = 10…640"),
        ("Q7", "Why not SS4?",        "Case 9 -- equal-time Pareto"),
        ("Q8", "Gaussian IC only?",   "Case 10 -- sech pulse"),
        ("Q9", "Lossless only?",      "Case 11 -- 0.2 dB/km lossy fiber"),
    ]
    for q, question, answer in checks:
        print(f"  [{q}] {question:25s} → {answer}")
    print(sep)

[Font] Serif family: DejaVu Serif
[Font] 'Times New Roman' not found; graceful fallback applied.
  DM SOLITON SSFM -- v4.3 REFEREE-PROOF VALIDATION SUITE

[GPU] NVIDIA GeForce RTX 5090 | 31.4 GB total | 0.04 GB free

  gamma=1.317e-03 1/(W*m)  |  L_D=5000 m  |  P0_N1=0.1518 W
  Figures -> /20140171/AI4Science/非线性光纤/figures_jcp/
--------------------------------------------------------------------------------
CASE 1: BASELINE SYMMETRIC MAP  N≈1.4
--------------------------------------------------------------------------------

[Case 1 (Baseline)]  Ltot=6000 m  L1=50  L2=50  P0=0.30 W  alpha=0.00e+00
  Ref (dz=0.01562 m) built in 138.1 s


Case 1 (Baseline): 100%|██████████████████| 4/4 [00:31<00:00,  7.91s/it]



      dz |   Err Trad |    p_T |    Err Mid |    p_M |  Spdup
  -------|------------|--------|------------|--------|-------
   2.000 |   8.92e-04 |    nan |   3.38e-07 |    nan |   1.01
   1.000 |   4.46e-04 |  1.000 |   8.46e-08 |  2.000 |   1.01
   0.500 |   2.23e-04 |  1.000 |   2.11e-08 |  2.001 |   1.01
   0.250 |   1.12e-04 |  1.000 |   5.27e-09 |  2.004 |   1.00
  => Trad avg order: 1.000  |  Mid avg order: 2.002
  => Error reduction @ dz=1.0: 5275x
[PANEL] -> ./figures_jcp/Fig01_Case1_Baseline_Panel.pdf
CASE 2: STRONG BREATHING  P0=0.60 W  N≈2.0
--------------------------------------------------------------------------------

[Case 2 (Strong)]  Ltot=2000 m  L1=50  L2=50  P0=0.60 W  alpha=0.00e+00
  Ref (dz=0.03125 m) built in 23.1 s


Case 2 (Strong): 100%|████████████████████| 3/3 [00:09<00:00,  3.32s/it]



      dz |   Err Trad |    p_T |    Err Mid |    p_M |  Spdup
  -------|------------|--------|------------|--------|-------
   1.000 |   2.68e-04 |    nan |   3.38e-08 |    nan |   1.01
   0.500 |   1.34e-04 |  1.000 |   8.43e-09 |  2.005 |   1.02
   0.250 |   6.70e-05 |  1.000 |   2.08e-09 |  2.018 |   1.01
  => Trad avg order: 1.000  |  Mid avg order: 2.011
  => Error reduction @ dz=0.5: 15888x
CASE 3: ASYMMETRIC MAP  60:40 DUTY CYCLE
--------------------------------------------------------------------------------

[Case 3 (60:40)]  Ltot=2000 m  L1=60  L2=40  P0=0.30 W  alpha=0.00e+00
  Ref (dz=0.03125 m) built in 23.5 s


...


  2nd-order results:
    dz=2.000  Trad=2.48e-04 (0.37s)  Mid=3.51e-08 (0.36s)  ratio=7043x
    dz=1.000  Trad=1.24e-04 (0.73s)  Mid=8.78e-09 (0.72s)  ratio=14101x
    dz=0.500  Trad=6.19e-05 (1.44s)  Mid=2.19e-09 (1.44s)  ratio=28309x
    dz=0.250  Trad=3.10e-05 (2.91s)  Mid=5.39e-10 (2.90s)  ratio=57370x
    dz=0.125  Trad=1.55e-05 (5.82s)  Mid=1.30e-10 (5.79s)  ratio=118663x

  SS4 results:
    dz=4.000  SS4=8.08e-03 (0.63s)  SS4_cost/Mid_cost≈3x
    dz=2.000  SS4=4.16e-11 (1.27s)  SS4_cost/Mid_cost≈3x
    dz=1.000  SS4=3.96e-11 (2.54s)  SS4_cost/Mid_cost≈3x
    dz=0.500  SS4=3.58e-11 (5.05s)  SS4_cost/Mid_cost≈3x
    dz=0.250  SS4=2.83e-11 (10.08s)  SS4_cost/Mid_cost≈3x
[PLOT] -> ./figures_jcp/Fig09_Case9_SS4_Comparison.pdf
CASE 10 (NEW): INITIAL CONDITION -- sech pulse (non-Gaussian)
--------------------------------------------------------------------------------

[Case 10 (sech IC)]  Ltot=2000 m  L1=50  L2=50  P0=0.30 W  alpha=0.00e+00
  Ref (dz=0.03125 m) built in 22.7 s


Case 10 (sech IC): 100%|██████████████████| 3/3 [00:10<00:00,  3.37s/it]



      dz |   Err Trad |    p_T |    Err Mid |    p_M |  Spdup
  -------|------------|--------|------------|--------|-------
   1.000 |   1.04e-04 |    nan |   9.54e-09 |    nan |   1.01
   0.500 |   5.18e-05 |  1.000 |   2.38e-09 |  2.005 |   1.01
   0.250 |   2.59e-05 |  1.000 |   5.87e-10 |  2.018 |   1.00
  => Trad avg order: 1.000  |  Mid avg order: 2.012
  => Error reduction @ dz=0.5: 21777x
CASE 11 (NEW): LOSSY FIBER  alpha=0.2 dB/km
--------------------------------------------------------------------------------
  alpha = 0.2 dB/km = 4.6052e-05 Np/m

[Case 11 (Lossy)]  Ltot=2000 m  L1=50  L2=50  P0=0.30 W  alpha=4.61e-05
  Ref (dz=0.03125 m) built in 23.1 s


Case 11 (Lossy): 100%|████████████████████| 3/3 [00:10<00:00,  3.36s/it]



      dz |   Err Trad |    p_T |    Err Mid |    p_M |  Spdup
  -------|------------|--------|------------|--------|-------
   1.000 |   1.18e-04 |    nan |   8.12e-09 |    nan |   1.00
   0.500 |   5.89e-05 |  1.000 |   2.02e-09 |  2.005 |   1.00
   0.250 |   2.95e-05 |  1.000 |   4.99e-10 |  2.019 |   1.00
  => Trad avg order: 1.000  |  Mid avg order: 2.012
  => Error reduction @ dz=0.5: 29137x
MERGED PANEL GENERATION
--------------------------------------------------------------------------------
[PANEL] -> ./figures_jcp/Fig02_Cases234_Convergence_Panel.pdf
[PANEL] -> ./figures_jcp/Fig10_Cases1011_Generalization_Panel.pdf
GLOBAL EFFICIENCY PARETO
[PLOT] -> ./figures_jcp/Fig11_Global_Pareto.pdf

✅  v4.3 REFEREE-PROOF VALIDATION COMPLETE

  All 8 figures (PDF, vector) in: /20140171/AI4Science/非线性光纤/figures_jcp/

                    Case |  p_Trad |   p_Mid | Referee question answered
  ----------------------------------------------------------------------
              1 Baseline |  